<a href="https://colab.research.google.com/github/hmmnyamminji/python/blob/main/WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 웹 스크래핑 (Web Scraping)

웹 스크래핑(Web Scraping) 또는 **웹 크롤링(Web Crawling)** 은 프로그램으로 웹사이트의 페이지를 옮겨 가면서 데이터를 추출하는 작업

* **뷰티풀숩(BeautifulSoup)**: HTML 안에서 원하는 태그나 텍스트를 찾을 때

* **HTML 파싱** - 복잡한 텍스트로 이루어진 HTML 코드를 웹 브라우저나 다른 프로그램이 이해하고 조작하기 쉬운 '객체'나 '트리 구조'로 바꾸는 작업



In [ ]:
import requests

# 삼성전자 종목 코드 005930
# https://finance.naver.com/item/main.naver?code=005930

# sk하이닉스 종목 코드 000660
# https://finance.naver.com/item/main.naver?code=000660


In [ ]:
code = "005930"
url = "https://finance.naver.com/item/main.naver?code=005930"

response = requests.get(url.format(code))
print(response.text) # 응답 HTML 출력



	
	
	
	
	
<html lang='ko'>
<head>


	
		<title>삼성전자 : Npay 증권</title>
	
	




<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />

<meta http-equiv="Content-Script-Type" content="text/javascript">
<meta http-equiv="Content-Style-Type" content="text/css">
<meta name="apple-mobile-web-app-title" content="Npay 증권" />





	
    
        <meta property="og:url" content="https://finance.naver.com/item/main.naver?code=005930"/>
        
			
		    
		    	<meta property="og:title" content="삼성전자 - Npay 증권 : Npay 증권"/>
		     
		
		
			
			   <meta property="og:description" content="관심종목의 실시간 주가를 가장 빠르게 확인하는 곳"/>
		    
		    
		
		 
			
			    <meta property="og:image" content="https://ssl.pstatic.net/static/m/stock/im/2016/08/og_stock-200.png"/>
		    
		    
		
    

<meta property="og:type" content="article"/>
<meta property="og:article:thumbnailUrl" content=""/>
<meta property="og:article:author" content="Npay 증권"/>
<meta property="og:article:author:url" content="http:/

In [ ]:
from bs4 import BeautifulSoup #HTML 및 XML 파일을 파싱

soup = BeautifulSoup(response.text, 'html.parser') #응답 HTML을 파싱하여 soup객체를 생성

In [ ]:
# 현재가 태그 찾기
price_tag = soup.find('p', attrs={'class': 'no_today'})
current_price = price_tag.find('span', attrs={'class': 'blind'}).get_text() # 숫자 텍스트만 추출
print(f"현재가: {current_price}")

현재가: 173,500


In [ ]:
# 시세 테이블 찾기
info_table = soup.find('table', attrs={'class': 'no_info'})

td_list = info_table.find_all('td') # 테이블의 모든 <td> 태그 추출

for td in td_list:
  label = td.find('span', attrs={'class': 'sptxt'}).get_text()
  value = td.find('span', attrs={'class': 'blind'}).get_text()
  if label and value:
      print(f"{label}: {value}")

전일: 188,200
고가: 175,500
거래량: 43,066,020
시가: 173,500
저가: 167,300
거래대금: 7,376,525


In [ ]:
# 쉼표 제거 후 정수로 변환
price = int(current_price.replace(',', ''))
print(price)

173500


In [ ]:
import requests
from bs4 import BeautifulSoup

# 함수 만들기
def get_stock_info(code):
  url = "https://finance.naver.com/item/main.naver?code={}"
  response = requests.get(url.format(code))
  response.encoding = "utf-8"
  soup = BeautifulSoup(response.text, 'html.parser')

  price_tag = soup.find('p', attrs={'class': 'no_today'})
  current_price = price_tag.find('span', attrs={'class': 'blind'}).get_text()

  info_table = soup.find('table', attrs={'class': 'no_info'})
  td_list = info_table.find_all('td')

  result = {'현재가': current_price}
  for td in td_list:
    label = td.find('span', attrs={'class': 'sptxt'}).get_text()
    value = td.find('span', attrs={'class':'blind'})
    if label and value:
      result[label] = value.get_text()

  return result

In [ ]:
samsung_info = get_stock_info('005930')
print(samsung_info)

{'현재가': '173,500', '전일': '188,200', '고가': '175,500', '거래량': '43,066,020', '시가': '173,500', '저가': '167,300', '거래대금': '7,376,525'}


In [ ]:
import pandas as pd

#종목 데이터프레임 생성
stocks_df = pd.DataFrame({
    'name' : ['삼성전자','SK하이닉스','현대차','한화시스템'],
    'code' : ['005930','000660','005380','272210']
})

def get_price(row):
  info = get_stock_info(row['code'])
  return info.get('현재가', None) # '현재가'를 반환, 없으면 None 반환

stocks_df['현재가'] = stocks_df.apply(get_price, axis=1) #데이터프레임의 각 행에 대해 함수 적용
# axis=1 -> 행

# 람다 함수로 간결하게
stocks_df['현재가'] = stocks_df.apply(
    lambda row: get_stock_info(row['code']).get('현재가', None), axis=1
)
display(stocks_df)

,name,code,현재가
0,삼성전자,005930,"173,500"
1,SK하이닉스,000660,"836,000"
2,현대차,005380,"507,000"
3,한화시스템,272210,"162,700"


In [ ]:
print(stocks_df)

     name    code
0    삼성전자  005930
1  SK하이닉스  000660
2     현대차  005380
3   한화시스템  272210
